In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import pickle

# Constants
M_sun = 2e33             # Solar mass in g
M_tot = 0.01 * M_sun     # Total mass of the ejecta in g
c = 3e10                 # Speed of light in cm/s
v_min = 0.1 * c          # Minimum velocity of massive shell in cm/s
beta = 3                 # factor for homologous expansion - ensures shells never overlap 
numshells = 10           # Number of shells
sigma = 5.67e-5          # Stefan-Boltzmann constant in erg/(cm^2 K^4 s)
h = 6.626e-27            # Planck's constant in erg s
nu = 1e14                # Frequency in Hz (arbitrary value for example)
D = 1e26                 # Distance in cm (arbitrary for example)
kB = 1.38e-16            # Boltzmann constant in erg/K

# Time array
t = np.logspace(0, 6, 50)              

# Velocity array for each shell
v_shells = np.linspace(0.1, 1, numshells) * c

# Mass distribution
M_shells = beta * M_tot * (v_min**(beta)) * (v_shells**(-beta-1))    # Mass fraction based on velocity
M_shells = M_shells * M_tot / np.sum(M_shells)                       # Normalised so the shells sum to M_0
E_int_shells = 0.5 * M_shells * (v_shells**2)                        # Internal energy per shell, assuming kinetic energy is similar to internal energy

# Dataframe of properties
shell_data = pd.DataFrame({'Velocity (cm/s)': v_shells, 'Mass (g)': M_shells, 'Internal Energy (erg)': E_int_shells})

# Constants associated with heating and radiative loss terms
av = 0.56
bv = 0.71
cv = 0.74
xr = 0.6
kv = [20, 5, 1]                  # Opacity ranges for kilonova colors

# Heating efficiency as a function of time
def eps_th_v(t):
    t_day = t / 86400  # Convert seconds to days
    exp_term = np.exp(-av * t_day)
    log_term = np.log(1 + 2 * bv * t_day**cv)
    result = 0.36 * (exp_term + (log_term / (2 * bv * t_day**cv)))
    return result

# R-process energy deposition rate
def e_dot_r(t):
    t0 = 1.3  
    sigma = 0.11  
    arctan_term = np.arctan((t - t0) / sigma)
    result = 4e18 * eps_th_v(t) * (0.5 - (1 / np.pi) * arctan_term)**1.3
    return result

# Heating due to radioactive decay
def Q_rv(Mv, t):
    return Mv * xr * e_dot_r(t)

# Diffusion time
def t_diff(M_shells, t):
    return ((M_shells**(4./3.)) * kv[1]) / (4 * np.pi * (M_tot**(1./3.)) * v_min * t * c)

# Radiative losses
def L_v(R, Eint, t_diff):
    t_lc = (R / c)                            # Light-crossing time
    return Eint / (t_diff + t_lc)

# ODEs governing energy and dynamics
def energy(y, t, M):
    Eint, R, v = y
    diff_time = t_diff(M, t)
    dEint = -((Eint / R) * v) + Q_rv(M, t) - L_v(R, Eint, diff_time)
    dR = v
    dv = Eint / (M * R)
    return [dEint, dR, dv]

# Function to pre-solve the ODEs for all shells and store results
def pre_solve_shells(shell_data, t):
    E_all, R_all, v_all = [], [], []
    for i, row in shell_data.iterrows():
        M = row['Mass (g)']
        Eint0 = row['Internal Energy (erg)']
        v0 = row['Velocity (cm/s)']
        R0 = 1e9  # Initial radius
        y0 = [Eint0, R0, v0]
        Edep = odeint(energy, y0, t, args=(M,))
        E_all.append(Edep[:, 0])  
        R_all.append(Edep[:, 1])  
        v_all.append(Edep[:, 2])
    
    return np.array(E_all), np.array(R_all), np.array(v_all)


# Pre-solve for all shells
E_all, R_all, v_all = pre_solve_shells(shell_data, t)


def compute_photosphere_radius(R_all, shell_masses, t):
    num_shells, num_times = R_all.shape
    R_ph = np.zeros(num_times)
    
    # Calculate tau values and cumulative optical depth for each time step in vectorized form
    tau_all = np.zeros_like(R_all)  # Array to store optical depth values per shell per time
    
    for i in range(num_shells):
        tau_all[i, :] = kv[2] * shell_masses[i] / (4 * np.pi * R_all[i, :]**2)
    
    tau_cumu = np.cumsum(tau_all[::-1], axis=0)[::-1]  # Cumulative sum in reverse order
    
    # Calculate R_ph based on cumulative optical depth and where tau >= 1
    for i in range(num_times):
        tau_cumu_at_t = tau_cumu[:, i]
        if tau_cumu_at_t[-1] < 1:
            R_ph[i] = R_all[-1, i]
        else:
            idx = np.where(tau_cumu_at_t >= 1)[0][0]
            if idx == 0:
                R_ph[i] = R_all[0, i]
            else:
                tau_low = tau_cumu_at_t[idx - 1]
                tau_high = tau_cumu_at_t[idx]
                R_low = R_all[idx - 1, i]
                R_high = R_all[idx, i]
                f = (1 - tau_low) / (tau_high - tau_low)
                R_ph[i] = R_low + f * (R_high - R_low)
    return R_ph

R_ph = compute_photosphere_radius(np.array(R_all), M_shells, t)

L_tot = np.sum([L_v(R, Eint, t_diff(shell_data['Mass (g)'][i], t)) 
                for i, (R, Eint) in enumerate(zip(R_all, E_all))], axis=0)

def calculate_flux(R_ph, L_tot):
    T_eff = (L_tot / (4 * np.pi * sigma * R_ph**2))**(1/4)
    
    Flux = ((2 * np.pi * h * nu**3) / (c**2)) * (1 / (np.exp((h * nu) / (kB * T_eff)) - 1)) * (R_ph**2 / D**2)
    
    return Flux

# Calculate the flux values
Flux_values = calculate_flux(R_ph, L_tot)

# Add noise to the flux
noise = 0.1 * Flux_values  # Noise scaling factor
Flux_perturbed = Flux_values + noise * np.random.randn(*Flux_values.shape)

with open('flux_data.pkl', 'wb') as f:
    pickle.dump(Flux_values, f)

In [ ]:
import emcee
import corner

def model(theta, t):
    log_M_tot, log_Eint0, R0, v0 = theta
    M_tot = 10**log_M_tot
    Eint0 = 10**log_Eint0
    # Regenerate shell masses and velocities
    v_shells = np.linspace(0.1, 1, numshells) * v0  # Scale shells based on v0 instead of fixed c
    M_shells = beta * M_tot * (v_min**(beta)) * (v_shells**(-beta-1))
    M_shells = M_shells * M_tot / np.sum(M_shells)  # Normalize
    
    E_int_shells = np.full_like(v_shells, Eint0 / numshells)  # Distribute Eint0 uniformly or based on M_shells
    
    # Build new shell data
    shell_data_dynamic = pd.DataFrame({
        'Velocity (cm/s)': v_shells,
        'Mass (g)': M_shells,
        'Internal Energy (erg)': E_int_shells
    })

    # Solve ODEs
    E_all, R_all, v_all = pre_solve_shells(shell_data_dynamic, t)
    R_ph = compute_photosphere_radius(R_all, M_shells, t)

    L_tot = np.sum([L_v(R, Eint, t_diff(M_shells[i], t)) 
                    for i, (R, Eint) in enumerate(zip(R_all, E_all))], axis=0)

    T_eff = (L_tot / (4 * np.pi * sigma * (R_ph**2)))**(1/4)
    exponent = (h * nu) / (kB * T_eff)
    Flux_model = ((2 * np.pi * h * (nu**3)) / (c**2)) * (1 / (np.exp(exponent) - 1)) * (R_ph**2 / D**2)
    return Flux_model


# Log likelihood function
def log_likelihood(theta, t, y, yerr):
    model_flux = model(theta, t)
    return -0.5 * np.sum(((y - model_flux) / yerr)**2)

def log_prior(theta):
    log_M_tot, log_Eint0, R0, v0 = theta
    if (27 < log_M_tot< 34 and     # Mass range
        45 < log_Eint0 < 51 and      # Internal energy range
        1e8 < R0 < 1e17 and          # Initial radius range
        1e8 < v0 < 1e11):            # Initial velocity range
        return 0.0                   # Uniform prior
    return -np.inf  

# Log posterior function
def log_posterior(theta, t, y, yerr):
    lp = log_prior(theta)
    if np.isinf(lp):
        return lp
    return lp + log_likelihood(theta, t, y, yerr)

# Define MCMC function WITHOUT parallelization
def run_mcmc(initial, t, Flux_perturbed, noise, nsteps=10000, nwalkers=100, ndim=4):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior, args=(t, Flux_perturbed, noise))
    sampler.run_mcmc(initial, nsteps, progress=True)
    return sampler

# Initialize walkers and sample the posterior
Eint0 = np.mean(E_int_shells)  
v0 = np.mean(v_shells)    
R0 = 1e9 
M_0 = np.mean(M_shells)

log_M_tot = np.log10(M_0)  # Logarithm of total mass for prior
log_Eint0 = np.log10(Eint0)  # Logarithm of initial internal energy for prior

mass_noise_scale = 0.1  
Eint0_noise_scale = 0.1  
v0_noise_scale = 0.05
R0_noise_scale = 0.05


ndim = 4
nwalkers = 100
nsteps = 10000

initial = np.array([log_M_tot, log_Eint0, R0, v0
]) + np.array([
    mass_noise_scale * log_M_tot, Eint0_noise_scale * log_Eint0,
    R0_noise_scale * R0, v0_noise_scale * v0
]) * np.random.randn(nwalkers, ndim)
print(f"Initial shape: {initial.shape}")

sampler = run_mcmc(initial, t, Flux_perturbed, noise)

# Extract samples after burn-in (discard first 1000 steps)
samples = sampler.get_chain(discard=1000, thin=10, flat=True)

# Plot the results
samples_linear = samples.copy()
samples_linear[:, 0] = 10**samples[:, 0]  # M_0
samples_linear[:, 1] = 10**samples[:, 1]  # Eint0
corner.corner(samples_linear, labels=["M_0", "Eint0", "R0", "v0"], 
              truths=[M_0, Eint0, R0, v0])

In [ ]:
print("Mean acceptance fraction:", np.mean(sampler.acceptance_fraction))

In [ ]:
for i in range(ndim):
    plt.plot(sampler.get_chain()[:, :, i])
    plt.title(f"Parameter {i}")
    plt.xlabel("Step")
    plt.ylabel(f"Theta_{i}")
    plt.show()


In [ ]:
try:
    tau = sampler.get_autocorr_time()
    print("Autocorrelation time:", tau)
except emcee.autocorr.AutocorrError:
    print("Chain too short to estimate autocorrelation time.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Initial conditions (y0)
y0 = [M_tot, Eint0, R0, v0]

# Create the plot
plt.figure(figsize=(8, 6))

# Plot the total flux (sum of all shells) with noise added (Flux_perturbed)
plt.loglog(t, Flux_perturbed, label='Flux (Perturbed)', color='green', alpha=0.7)

# Plot the model flux (using y0 as the initial parameters)
plt.loglog(t, model(y0, t), label='Model Flux (Initial)', color='blue', linestyle='--')

# Plot the model flux for a range of random samples from the MCMC chain
for theta in samples[np.random.randint(len(samples), size=100)]:
    plt.plot(t, model(theta, t), color="red", alpha=0.05)

# Add labels and title
plt.xlabel('Time (s)')
plt.ylabel('Flux (erg/cm^2/s)')
plt.xlim([1e3, 1e6])
plt.ylim([1e-33, 1e-26])
plt.title('Flux Evolution over Time')

# Display grid and legend
plt.grid(True)
plt.legend()

# Show the plot
plt.show()

In [3]:
import emcee
import corner

def model(theta, t):
    log_M_tot, log_Eint0, log_R0, log_v0 = theta
    M_0 = 10**log_M_tot
    Eint0 = 10**log_Eint0
    R0 = 10**log_R0
    v0 = 10**log_v0

    # Regenerate shell masses and velocities
    v_shells = np.linspace(0.1, 1, numshells) * v0
    M_shells = beta * M_0 * (v_min**beta) * (v_shells**(-beta - 1))
    M_shells *= M_0 / np.sum(M_shells)

    E_int_shells = np.full_like(v_shells, Eint0 / numshells)

    shell_data_dynamic = pd.DataFrame({
        'Velocity (cm/s)': v_shells,
        'Mass (g)': M_shells,
        'Internal Energy (erg)': E_int_shells
    })

    E_all, R_all, v_all = pre_solve_shells(shell_data_dynamic, t)
    R_ph = compute_photosphere_radius(R_all, M_shells, t)

    L_tot = np.sum([
        L_v(R, Eint, t_diff(M_shells[i], t)) 
        for i, (R, Eint) in enumerate(zip(R_all, E_all))
    ], axis=0)

    T_eff = (L_tot / (4 * np.pi * sigma * (R_ph**2)))**(1/4)
    exponent = (h * nu) / (kB * T_eff)

    Flux_model = ((2 * np.pi * h * (nu**3)) / (c**2)) * \
                 (1 / (np.exp(exponent) - 1)) * (R_ph**2 / D**2)
    return Flux_model


def log_likelihood(theta, t, y, yerr):
    try:
        model_flux = model(theta, t)
        if not np.all(np.isfinite(model_flux)):
            return -np.inf
        return -0.5 * np.sum(((y - model_flux) / yerr) ** 2)
    except Exception:
        return -np.inf


def log_prior(theta):
    log_M_tot, log_Eint0, log_R0, log_v0 = theta
    if (27 < log_M_tot < 34 and      # Mass: 1e27–1e34 g
        45 < log_Eint0 < 51 and      # Energy: 1e45–1e51 erg
        8 < log_R0 < 17 and          # Radius: 1e8–1e17 cm
        8 < log_v0 < 11):            # Velocity: 1e8–1e11 cm/s
        return 0.0
    return -np.inf


def log_posterior(theta, t, y, yerr):
    lp = log_prior(theta)
    if np.isinf(lp):
        return lp
    return lp + log_likelihood(theta, t, y, yerr)


def run_mcmc(initial, t, Flux_perturbed, noise, nsteps=10000, nwalkers=100, ndim=4):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior, args=(t, Flux_perturbed, noise))
    sampler.run_mcmc(initial, nsteps, progress=True)
    return sampler


# === Initialization ===
Eint0 = np.mean(E_int_shells)
v0 = np.mean(v_shells)
R0 = 1e9
M_0 = np.mean(M_shells)

# Convert to log-space
log_M_tot = np.log10(M_0)
log_Eint0 = np.log10(Eint0)
log_R0 = np.log10(R0)
log_v0 = np.log10(v0)

ndim = 4
nwalkers = 100
nsteps = 10000

# Noise scales (log-space)
log_noise_scales = np.array([
    0.1 * log_M_tot,
    0.1 * log_Eint0,
    0.05 * log_R0,
    0.05 * log_v0
])

initial = np.array([log_M_tot, log_Eint0, log_R0, log_v0]) + \
          log_noise_scales * np.random.randn(nwalkers, ndim)

print(f"Initial shape: {initial.shape}")

sampler = run_mcmc(initial, t, Flux_perturbed, noise)

# === Extract and plot results ===
samples = sampler.get_chain(discard=1000, thin=10, flat=True)

# Transform samples back to linear space for plotting
samples_linear = samples.copy()
samples_linear[:, 0] = 10**samples[:, 0]  # M_0
samples_linear[:, 1] = 10**samples[:, 1]  # Eint0
samples_linear[:, 2] = 10**samples[:, 2]  # R0
samples_linear[:, 3] = 10**samples[:, 3]  # v0

corner.corner(samples_linear, labels=["M_0", "Eint0", "R0", "v0"],
              truths=[M_0, Eint0, R0, v0])


Initial shape: (100, 4)


C:\Users\olibr\AppData\Local\Temp\ipykernel_1560\2713482712.py:36: RuntimeWarning: overflow encountered in exp
  (1 / (np.exp(exponent) - 1)) * (R_ph**2 / D**2)
  0%|          | 0/10000 [00:00<?, ?it/s]c:\Users\olibr\pythonconda\Lib\site-packages\emcee\moves\red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
  0%|          | 2/10000 [00:06<9:01:11,  3.25s/it]Traceback (most recent call last):
  File "c:\Users\olibr\pythonconda\Lib\site-packages\emcee\ensemble.py", line 640, in __call__
    return self.f(x, *self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\olibr\AppData\Local\Temp\ipykernel_1560\2713482712.py", line 64, in log_posterior
    return lp + log_likelihood(theta, t, y, yerr)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\olibr\AppData\Local\Temp\ipykernel_1560\2713482712.py", line 42, in log_likelihood
    model_flux = model(theta, t)
               

emcee: Exception while calling your likelihood function:
  params: [31.60326284 49.79771119  8.94493927  9.52551747]
  args: (array([1.00000000e+00, 1.32571137e+00, 1.75751062e+00, 2.32995181e+00,
       3.08884360e+00, 4.09491506e+00, 5.42867544e+00, 7.19685673e+00,
       9.54095476e+00, 1.26485522e+01, 1.67683294e+01, 2.22299648e+01,
       2.94705170e+01, 3.90693994e+01, 5.17947468e+01, 6.86648845e+01,
       9.10298178e+01, 1.20679264e+02, 1.59985872e+02, 2.12095089e+02,
       2.81176870e+02, 3.72759372e+02, 4.94171336e+02, 6.55128557e+02,
       8.68511374e+02, 1.15139540e+03, 1.52641797e+03, 2.02358965e+03,
       2.68269580e+03, 3.55648031e+03, 4.71486636e+03, 6.25055193e+03,
       8.28642773e+03, 1.09854114e+04, 1.45634848e+04, 1.93069773e+04,
       2.55954792e+04, 3.39322177e+04, 4.49843267e+04, 5.96362332e+04,
       7.90604321e+04, 1.04811313e+05, 1.38949549e+05, 1.84206997e+05,
       2.44205309e+05, 3.23745754e+05, 4.29193426e+05, 5.68986603e+05,
       7.54312006e+05,

KeyboardInterrupt: 